#Assignment 2
##  Task 2: Recommender System using PySpark
### Introduction
The second task focused on building a recommendation model using machine‑learning. After preparing and analysing the data, I trained an ALS model to generate personalised game recommendations for each user using various hyperparameters.

### Data Import and Preprocessing
The dataset was loaded into a PySpark DataFrame. The header option was set to false because the file did not include a header row, and inferSchema was set to true so Spark could automatically detect the correct data types. After loading the data, I displayed the DataFrame to confirm that all columns were read correctly and assigned appropriate types. Since there was no header row, I then renamed the default column names to more appropriate ones.




In [0]:
#Reads the csv file into a dataframe
df2 = spark.read.csv(
"/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv",
header=False, # Use the first row as the header
inferSchema=True, # Infer data types
quote='"', # Define the quote character
escape='"', # Escape quotes inside quoted fields
multiLine=True # Enable multiline support
)
#Displays the data in the dataframe
display(df2.limit(10))

_c0,_c1,_c2,_c3
151603712,The Elder Scrolls V Skyrim,purchase,1.0
151603712,The Elder Scrolls V Skyrim,play,273.0
151603712,Fallout 4,purchase,1.0
151603712,Fallout 4,play,87.0
151603712,Spore,purchase,1.0
151603712,Spore,play,14.9
151603712,Fallout New Vegas,purchase,1.0
151603712,Fallout New Vegas,play,12.1
151603712,Left 4 Dead 2,purchase,1.0
151603712,Left 4 Dead 2,play,8.9


In [0]:
#Renamed the columns of the dataframe
df2 = (
    df2.withColumnRenamed("_c0", "ID")
      .withColumnRenamed("_c1", "Game Name")
      .withColumnRenamed("_c2", "Behaviour")
      .withColumnRenamed("_c3", "Hours Played")
)
#Displays the dataframe with the updated column names
display(df2.limit(10))

ID,Game Name,Behaviour,Hours Played
151603712,The Elder Scrolls V Skyrim,purchase,1.0
151603712,The Elder Scrolls V Skyrim,play,273.0
151603712,Fallout 4,purchase,1.0
151603712,Fallout 4,play,87.0
151603712,Spore,purchase,1.0
151603712,Spore,play,14.9
151603712,Fallout New Vegas,purchase,1.0
151603712,Fallout New Vegas,play,12.1
151603712,Left 4 Dead 2,purchase,1.0
151603712,Left 4 Dead 2,play,8.9


## Exploratory Data Analysis
In this section, I carried out several exploratory checks to understand the structure and behaviour of the data before moving on to modelling. I first examined the distinct number of games and how often each one appeared with COUNT(), which helped identify the most common titles in the data. I then looked at how many times Play and Purchased appeared, to get a sense of what majority of users were doing
I was able to find the average hours played using the filtered "Play" behaviour, this gave me an insight to the typical amount of time users spent on games they played
To understand user engagement, I calculated the average hours played by filtering for entries where the behaviour was “play”. Additionally, I identified the top and bottom games based on total hours played. I grouped the behaviour "Played" by game, calculated the total hours, and ordered the results using DESC() and ASC() to highlight the top and least played games..


In [0]:
import pyspark.sql.functions as F
# Distinct users and games
#Displays the distinct Game Names and the number of times they appear
df2.groupBy("Game Name").count().show()

# Behaviour distribution
#Displays the data in the Behaviour column (Whether a Game was played or purchased) and the number of times they appear
df2.groupBy("Behaviour").count().show()

# Average hours Played
# Extracts when Behaviour = played (When a game was played) agrregates the corresponding hours played from the Hours played column finds and displays the average hours played
df2.filter(F.col("Behaviour") == "play") \
  .agg(F.avg("Hours Played").alias("Average Hours Played")) \
  .show()
  
#Top 10 games by hours played
#Find the total hours played and displays them in descending order to find the top 10 games by hours played
df2.filter(F.col("Behaviour") == "play") \
  .groupBy("Game Name") \
  .agg(F.sum("Hours Played").alias("Total Hours Played")) \
  .orderBy(F.desc("Total Hours Played")) \
  .limit(10) \
  .show()

# Bottom 10 games by hours played
#Find the total hours played and displays them in ascending order to find the bottom 10 games by hours played
df2.filter(F.col("Behaviour") == "play") \
  .groupBy("Game Name") \
  .agg(F.sum("Hours Played").alias("Total Hours Played")) \
  .orderBy(F.asc("Total Hours Played")) \
  .limit(10) \
  .show()



+--------------------+-----+
|           Game Name|count|
+--------------------+-----+
|              Dota 2| 9682|
|METAL GEAR SOLID ...|  134|
|LEGO Batman The V...|   17|
|                RIFT|  236|
|             Anodyne|   14|
|  Legend of Grimrock|   93|
|Divinity Original...|  138|
|            Meltdown|    4|
|SanctuaryRPG Blac...|    6|
|       Snuggle Truck|   14|
|        Lunar Flight|   16|
|          Dungeons 2|   12|
|      Zuma's Revenge|    7|
|         HassleHeart|    1|
|Ihf Handball Chal...|    1|
|NEON STRUCT Sound...|    2|
|Dota 2 - The Inte...|    5|
|Agarest - Unlock ...|    1|
|RPG Maker Arabian...|    1|
|  The Black Watchmen|    2|
+--------------------+-----+
only showing top 20 rows
+---------+------+
|Behaviour| count|
+---------+------+
| purchase|129511|
|     play| 70489|
+---------+------+

+--------------------+
|Average Hours Played|
+--------------------+
|  48.878063243911484|
+--------------------+

+--------------------+------------------+
|     

In [0]:
# Finds the top 5 games by hours played
#Filters out when Behaviour = play (When a game was played) and groups the data by Game Name and finds the total hours played and displays them in descending order to find the top 5 games by hours played
df2.filter(F.col("Behaviour") == "play") \
  .groupBy("Game Name") \
  .agg(F.sum("Hours Played").alias("Total Hours Played")) \
  .orderBy(F.desc("Total Hours Played")) \
  .limit(5) \
  .show()

  # Bottom 5 games by hours played
  #Same as above however the results are in ascending order to find the bottom 5 games by hours played
df2.filter(F.col("Behaviour") == "play") \
  .groupBy("Game Name") \
  .agg(F.sum("Hours Played").alias("Total Hours Played")) \
  .orderBy(F.asc("Total Hours Played")) \
  .limit(5) \
  .show()

+--------------------+------------------+
|           Game Name|Total Hours Played|
+--------------------+------------------+
|              Dota 2| 981684.5999999989|
|Counter-Strike Gl...|322771.60000000015|
|     Team Fortress 2|173673.29999999993|
|      Counter-Strike|          134261.1|
|Sid Meier's Civil...| 99821.29999999999|
+--------------------+------------------+

+--------------------+------------------+
|           Game Name|Total Hours Played|
+--------------------+------------------+
|Your Doodles Are ...|               0.1|
|    Unity of Command|               0.1|
|Agricultural Simu...|               0.1|
|        Real Warfare|               0.1|
|      Carnage Racing|               0.1|
+--------------------+------------------+



## Exploratory Data Analysis
### Results
By finding the top 5 games played I could see which games have the highest engagement, while the bottom 5 showed me the games with minimal playtime.

In [0]:
from pyspark.ml.feature import StringIndexer
import mlflow
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR) #The logging level for MLflow is set to show only errors
mlflow.pyspark.ml.autolog() #MLflow autologging is enabled for PySpark models
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb = ctx.notebookPath().get().split("/")[2] #The notebook name is taken from the current Databricks context
safe_path = f"dbfs:/FileStore/{nb}/" #A safe path is created for storing files related to this notebook
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc") #MLflow is set to use Databricks for tracking and the model registry
experiment_name = f"/Shared/{nb}_recommender_system" # The experiment name is created using the notebook name
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) #The experiment is created if it does not already exist
mlflow.set_experiment(experiment_name) #The experiment is set as the active one

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/831103309676161', creation_time=1773245135721, experiment_id='831103309676161', last_update_time=1776890156545, lifecycle_stage='active', name='/Shared/odl_user_2029830@databrickslabs.com_recommender_system', tags={'mlflow.experiment.sourceName': '/Shared/odl_user_2029830@databrickslabs.com_recommender_system',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'Group-17',
 'mlflow.ownerId': '82053111295162'}>

##### Generating ID values for Users and Games
To use the recommendation model, ALS cannot work with string based user or item identifiers, so I needed numeric ID values for both the users and the games. To convert these categorical identifiers, I transformed the original ID and Game Name fields into numerical index values using StringIndexer(), which assigns a unique integer to each distinct user and game. I applied the indexers using fit() and transform() so the mapping was derived from the dataset and then applied consistently to create the new userId and GameId columns. Since many recommendation systems require intege -based IDs rather than floating point values, both columns were cast to integer types. During training, casting also aids in preventing type related mistakes.


In [0]:
from pyspark.sql.types import IntegerType
# User and game identifiers are converted into numeric index values
UserID = StringIndexer(inputCol="ID", outputCol="userId")
GamesID = StringIndexer(inputCol="Game Name", outputCol="GameId")
#Dataframe changed to a nore appropriate name
df_Games = df2
#The string indexers are fitted and applied to create the new numeric columns
df_Games =UserID.fit(df_Games).transform(df_Games)
df_Games = GamesID.fit(df_Games).transform(df_Games)
# The ID columns are casted to integer type 
df_Games = df_Games.withColumn("userId", F.col("userId").cast(IntegerType())) \
       .withColumn("GameId", F.col("GameId").cast(IntegerType()))

In [0]:
#Displays the first 10 rows of the dataframe with the new ID's
display(df_Games.limit(10))

ID,Game Name,Behaviour,Hours Played,userId,GameId
151603712,The Elder Scrolls V Skyrim,purchase,1.0,635,8
151603712,The Elder Scrolls V Skyrim,play,273.0,635,8
151603712,Fallout 4,purchase,1.0,635,100
151603712,Fallout 4,play,87.0,635,100
151603712,Spore,purchase,1.0,635,332
151603712,Spore,play,14.9,635,332
151603712,Fallout New Vegas,purchase,1.0,635,29
151603712,Fallout New Vegas,play,12.1,635,29
151603712,Left 4 Dead 2,purchase,1.0,635,4
151603712,Left 4 Dead 2,play,8.9,635,4


##### Rating Column Added
A rating column was added to give each game a consistent score based on hours played, making it easier to compare games and use the values in the recommendation model.

In [0]:
#Creates a rating column and assigns a score based on hours played
df_Games = df_Games.withColumn(
    "Rating",
    F.when(F.col("Hours Played") == 1, 5)  
     .otherwise(
         20 + 5 * F.floor((F.col("Hours Played") - 15) / 50)
     )
)
#Displays the top 10 rows of the dataframe including the new rating column
display(df_Games.limit(10))


ID,Game Name,Behaviour,Hours Played,userId,GameId,Rating
151603712,The Elder Scrolls V Skyrim,purchase,1.0,635,8,5
151603712,The Elder Scrolls V Skyrim,play,273.0,635,8,45
151603712,Fallout 4,purchase,1.0,635,100,5
151603712,Fallout 4,play,87.0,635,100,25
151603712,Spore,purchase,1.0,635,332,5
151603712,Spore,play,14.9,635,332,15
151603712,Fallout New Vegas,purchase,1.0,635,29,5
151603712,Fallout New Vegas,play,12.1,635,29,15
151603712,Left 4 Dead 2,purchase,1.0,635,4,5
151603712,Left 4 Dead 2,play,8.9,635,4,15


## Model Training
The dataset was split into training and test sets, so the ALS model could be trained and evaluate the unseen data. I used an 80/20 split, so 80% of the data was used to train the model and the remaining 20% was used for testing. A fixed seed of 100 was applied to ensure the split can reproduced. 

In [0]:
# The dataset is split into training and test sets
#80% is used for training and 20% for testing
(training, test) = df_Games.randomSplit([0.8, 0.2], seed=100)

## Model Training
### Results
I was able to create the ALS recommendation model using the cleaned data. The data was split into training and test sets, and the ALS algorithm learned the patterns between users, games, and their ratings. The model was able to fit the training data without any issues. The learned user item relationships are stored to generating predictions. 

## Hyperparameter Tuning
Before training the model, I set up small lists of hyperparameters that I wanted to test, I focesed on values that made sense for my dataset, such as different ranks, regularisation strengths, and iteration counts. These were the parameters I was most interested in comparing. I then looped through every combination so each one became its own experiment run, and logged the values in MLflow so I could review the results properly later. Inside the loop, I created the ALS model using the specific rank, regularisation, and iteration values for that run, along with the user, item, and rating columns. These control how the model learns the user game patterns and overfitting. Once the model was defined, I fitted it on the training data so it could learn the relationships it needed to make recommendations.

In [0]:
# lists of hyperparameters to test
ranks = [5, 10] #Controls the number of latent factors
regParams = [0.01, 0.1] #Prevents Overfitting
maxIters = [5, 10] #Number of training iterations

# I loop through every combination of hyperparameters and train a model for each combination
for r in ranks:
    for reg in regParams:
        for it in maxIters:

            # New MLflow run so the parameters and results are tracked separately
            with mlflow.start_run():

                # Each hyperparameters is logged
                mlflow.log_param("regParam", reg)
                mlflow.log_param("maxIter", it)
                mlflow.log_param("rank", r)

In [0]:
from pyspark.ml.recommendation import ALS
# The ALS model is created using the rank, iteration, regularisation,
# and the users, items, and ratings columns
als=ALS(rank=r, maxIter=it, regParam=reg, userCol="userId", itemCol="GameId", ratingCol="Rating", seed=100)
# The model is fitted using the training dataset
model = als.fit(training)

## Hyperparameter Tuning
### Results
I ran the hyperparameter tuning step on the ALS model a few times using different combinations of rank, regularisation strength, and iteration counts. By testing these variations, I was able to see how each setting affected the model’s behaviour and stability. Across successfully without any errors, each run was logged in MLflow so I could compare the RMSE scores if I wanted too. The results showed clear differences in performance depending on the parameter values, which helped me understand how strongly the model handled overfitting and how quickly it learned user game patterns. Overall, the tuning process gave me a much clearer sense of which settings produced reliable recommendations and which combinations were less effective.

## Model Evaluation
To evaluate the model, I generated predictions on the test data and removed any rows with missing values so the results were based only on valid outputs. I then used a RegressionEvaluator to calculate the RMSE, which compares the model’s predicted ratings with the actual user ratings. This gave me a clear measure of how well the ALS model performed on unseen data, with lower RMSE values indicating better accuracy. For each run, the RMSE score was printed alongside the hyperparameters used, which helped me understand how different settings affected performance. I also displayed the prediction output so I could see the estimated ratings the model produced for each user game pair in the test set.


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator
# Predictions are created for the test dataset and any rows with missing values arepredictions = model.transform(test).dropna() 
predictions = model.transform(test).dropna() 
# The Root mean squared error (RMSE) is calculated using the Rating column and the model's prediction column
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="Rating",
    predictionCol="prediction"
)
# The RMSE value is calculated and displayed
rmse = evaluator.evaluate(predictions)
print(f"Run Completed rank={r}, reg={reg}, iter={it}, RMSE={rmse}")
#Displays the prediction results
predictions.show() 

Run Completed rank=10, reg=0.1, iter=10, RMSE=22.171358047078115
+-----+--------------------+---------+------------+------+------+------+----------+
|   ID|           Game Name|Behaviour|Hours Played|userId|GameId|Rating|prediction|
+-----+--------------------+---------+------------+------+------+------+----------+
| 5250|     Cities Skylines| purchase|         1.0|  1420|   158|     5| 24.567114|
| 5250|              Dota 2|     play|         0.2|  1420|     0|    15| 5.0879574|
| 5250|Half-Life 2 Death...| purchase|         1.0|  1420|    13|     5|  6.752001|
| 5250|              Portal| purchase|         1.0|  1420|    14|     5|  8.750904|
| 5250|     Team Fortress 2| purchase|         1.0|  1420|     1|     5| 14.386684|
|76767|         Alien Swarm|     play|         0.8|   756|    32|    15|  9.463665|
|76767|            Banished|     play|        24.0|   756|   214|    20| 18.357143|
|76767|            Banished| purchase|         1.0|   756|   214|     5| 18.357143|
|76767|Call

## Model Evaluation
### Results
The evaluation stage focused on assessing how well the trained ALS model performed on data it hadn’t seen before. The RMSE value produced was 22.17, this provided a clear indication of the model’s overall accuracy and how reliably it could estimate user preferences across different user game pair


## Generating Recommendations
Five game recommendations were generated for every user, using the trained ALS model. The output initially contained only user IDs, game IDs, and predicted scores, so I created a lookup table to map each GameId back to it's corresponding game name. The recommendation list was then exploded so that each recommended game appears on its own row, making the results easier to read and analyse.

In [0]:
# 5 game recommendations are generated for every user in the dataset
GameRecs= model.recommendForAllUsers(5)
#Displays the UserID with their recommendations
GameRecs.show()

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{265, 37.72072},...|
|    12|[{262, 41.648067}...|
|    13|[{4178, 54.180584...|
|    22|[{4178, 54.80722}...|
|    26|[{4178, 51.35441}...|
|    27|[{4178, 53.048817...|
|    28|[{3964, 76.207596...|
|    31|[{4178, 48.875908...|
|    34|[{4178, 38.051903...|
|    44|[{4178, 59.708813...|
|    47|[{4178, 67.13034}...|
|    52|[{4178, 43.74211}...|
|    53|[{4178, 49.449856...|
|    65|[{4178, 51.855164...|
|    76|[{805, 67.12111},...|
|    78|[{3964, 52.39488}...|
|    81|[{262, 171.50845}...|
|    85|[{262, 187.95961}...|
|    91|[{4178, 76.68657}...|
|    93|[{262, 342.4628},...|
+------+--------------------+
only showing top 20 rows


In [0]:
# Maps itemId back to the Game Name
lookup = df_Games.select("GameId", "Game Name").distinct()
# The recommendation list is expanded so each recommended game appears on its own row.
# The GameId and score are taken from the exploded structure.
# A lookup table is joined to add the game names.
final_recs = (
    GameRecs
    .select("userId", F.explode("recommendations").alias("rec"))
    .select("userId", F.col("rec.GameId").alias("GameId"), F.col("rec.rating").alias("Score"))
    .join(lookup, "GameId", "left")
)
# The first 5 rows are shown
final_recs.orderBy("userId", F.desc("score")).show(5, truncate=False)

+------+------+---------+----------------------------------------------+
|GameId|userId|Score    |Game Name                                     |
+------+------+---------+----------------------------------------------+
|4178  |0     |65.01922 |NOBUNAGA'S AMBITION Kakushin with Power Up Kit|
|3230  |0     |41.402973|Out of the Park Baseball 16                   |
|262   |0     |35.79209 |Football Manager 2014                         |
|2     |0     |32.09188 |Counter-Strike Global Offensive               |
|1447  |0     |31.901943|Guild Wars                                    |
+------+------+---------+----------------------------------------------+
only showing top 5 rows


## General Recommendation
### Results
Personalised game suggestions were generated for every user in the dataset. The model returned the top five games for each user based on their predicted rating scores, and each user was shown alongside their recommended games and the corresponding prediction score.

## Conclusion – Task 2: Recommender System
Overall, the recommender system was successfully built using Spark and MLlib, and the ALS model was able to pick up useful patterns in user game relationships. Working through the task helped me see how different hyperparameters influence performance and how Spark can scale recommendation workflows effectively. After only working with single parameter examples in the labs, it was interesting to observe how much the results changed when tuning multiple settings together.